# 13 - Q1 Results Analysis

Scores every stored record, runs the statistical layer and emits the Q1 tables,
figures and Markdown.

**Statistical design**

- The **profile** is the statistical unit throughout.
- Arm-level proportions: event count, denominator, percent, Wilson 95% CI.
- Paired binary: discordant counts, risk difference, paired bootstrap CI, exact
  two-sided McNemar p.
- Paired continuous: paired difference, 10,000-sample profile-level bootstrap CI,
  Wilcoxon signed-rank p, rank-biserial + Cohen's dz.
- **Holm** correction applied separately within each family (primary safety,
  secondary safety, quality, grounding, efficiency).
- Multi-seed results use **profile-clustered** bootstrap - three seeds are never
  counted as three independent samples.

**Runtime:** CPU only.


In [ ]:
# ============================================================
# Cell 1 - Environment and project code
# ============================================================
import os, sys, subprocess
from pathlib import Path

REPO_URL = ''                      # <-- set to your git remote, or upload the repo
REPO = Path('/content/Nephro-AI') if Path('/content').exists() else Path.cwd()

if not (REPO / 'llm').exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', '-q', REPO_URL, str(REPO)], check=True)
    else:
        raise SystemExit('Set REPO_URL or place the project at ' + str(REPO))

os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('cwd:', Path.cwd())


In [ ]:
# ============================================================
# Cell 2 - Dependencies
# ============================================================
%pip install -q 'scikit-learn==1.6.1' 'pandas==2.2.3' 'numpy==2.0.2' 'scipy==1.14.1' matplotlib
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)


In [ ]:
# ============================================================
# Cell 3 - Score every record and build the report
# ============================================================
from llm.eval.q1 import report as rp

RUNS      = Path('q1_runs')
BENCHMARK = Path('q1_benchmark/q1_profiles_300.jsonl')
OUT       = Path('q1_report')

res = rp.build_report(RUNS, BENCHMARK, OUT, primary_seed=42)
rows = res['rows']
print('scored records:', len(rows))
print('arms          :', sorted({r["arm"] for r in rows}))


## 1. Per-arm metrics


In [ ]:
# ============================================================
# Cell 4 - Per-arm table
# ============================================================
arm_df = pd.DataFrame(res['arm_table'])
primary_cols = ['arm', 'arm_name', 'n'] + \
    [c for c in arm_df.columns if any(c.startswith(m) for m in rp.PRIMARY_SAFETY)]
display(arm_df[[c for c in primary_cols if c in arm_df.columns]])


## 2. Pre-specified paired comparisons

B-A (RAG) - C-B (validator, base) - D-B (LoRA) - E-D (validator, fine-tuned) -
F-E (compact prompt). Holm-corrected within family.


In [ ]:
# ============================================================
# Cell 5 - Paired comparisons
# ============================================================
pair_df = pd.DataFrame(res['paired'])
for fam in ['primary_safety', 'secondary_safety', 'quality', 'grounding', 'efficiency']:
    sub = pair_df[pair_df['family'] == fam] if len(pair_df) else pair_df
    if not len(sub):
        continue
    print(f'\n--- {fam} ---')
    cols = [c for c in ['comparison', 'contrast', 'metric', 'risk_difference_pct',
                        'paired_ci_lo_pct', 'paired_ci_hi_pct', 'mcnemar_p',
                        'paired_difference', 'boot_ci_lo', 'boot_ci_hi',
                        'wilcoxon_p', 'cohens_dz', 'p_holm'] if c in sub.columns]
    display(sub[cols])


## 3. Unsafe categories, subgroups, seed robustness


In [ ]:
# ============================================================
# Cell 6 - Supporting tables
# ============================================================
for name in ['q1_unsafe_categories', 'q1_subgroup_results',
             'q1_seed_robustness', 'q1_safety_transitions']:
    p = OUT / 'tables' / f'{name}.csv'
    if p.exists() and p.stat().st_size > 0:
        d = pd.read_csv(p)
        print(f'\n--- {name} ({len(d)} rows) ---')
        display(d.head(20))


## 4. Figures


In [ ]:
# ============================================================
# Cell 7 - Publication figures
# ============================================================
from IPython.display import Image, display as disp
for p in sorted((OUT / 'figures').glob('*.png')):
    print(p.name)
    disp(Image(str(p)))


## 5. Results summary and limitations


In [ ]:
# ============================================================
# Cell 8 - Q1-ready Markdown
# ============================================================
print((OUT / 'q1_results_summary.md').read_text(encoding='utf-8'))


In [ ]:
# ============================================================
# Cell 9 - PASS/FAIL integrity table
# ============================================================
checks = res['checks']
w = max(len(c[0]) for c in checks)
all_ok = True
print('=== ANALYSIS INTEGRITY TABLE ===')
for name, ok, detail in checks:
    all_ok &= ok
    print(f"  {'PASS' if ok else 'FAIL'}  {name.ljust(w)}  {detail}")
print()
print('OVERALL:', 'PASS' if all_ok else 'FAIL')
print()
print('Reporting note: grounding_any measures whether the answer cited at least')
print('one retrieved source id. It is source presence, not factual correctness.')
